In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_108_Aya_Nagar_Delhi_IMD_1Day.csv")

In [3]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,141.20,309.48,37.27,26.23,63.19,NaN,NaN,1.84,33.93,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
1,2024-01-02,144.70,302.31,30.09,23.02,51.86,NaN,NaN,1.46,33.62,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2,2024-01-03,150.21,347.64,22.22,27.46,47.97,NaN,NaN,1.74,24.65,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
3,2024-01-04,142.34,331.43,28.93,23.24,52.09,NaN,NaN,1.64,11.86,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
4,2024-01-05,95.54,246.79,32.91,23.13,56.04,NaN,NaN,1.61,13.14,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,122.81,126.14,31.22,18.32,35.13,NaN,NaN,1.02,39.68,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
362,2024-12-28,68.66,104.18,12.10,13.99,17.27,NaN,NaN,0.31,32.91,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
363,2024-12-29,62.89,71.82,9.23,7.03,11.20,NaN,NaN,0.19,26.40,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
364,2024-12-30,66.95,101.48,6.57,8.78,9.98,NaN,NaN,0.23,52.08,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 13)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['NH3 (µg/m³)', 'Benzene (µg/m³)', 'Toluene (µg/m³)']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 Timestamp         0
PM2.5 (µg/m³)     0
PM10 (µg/m³)      0
NO (µg/m³)        0
NO2 (µg/m³)       0
NOx (ppb)         0
CO (mg/m³)        0
Ozone (µg/m³)     0
Xylene (µg/m³)    0
TOT-RF (mm)       0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (366, 10)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         141.20        309.48       37.27        26.23   
1  2024-01-02         144.70        302.31       30.09        23.02   
2  2024-01-03         150.21        347.64       22.22        27.46   
3  2024-01-04         142.34        331.43       28.93        23.24   
4  2024-01-05          95.54        246.79       32.91        23.13   

   NOx (ppb)  CO (mg/m³)  Ozone (µg/m³)  Xylene (µg/m³)  TOT-RF (mm)  
0      63.19        1.84          33.93          23.175          0.0  
1      51.86        1.46          33.62          23.175          0.0  
2      47.97        1.74          24.65          23.175          0.0  
3      52.09        1.64          11.86          23.175          0.0  
4      56.04        1.61          13.14          23.175          0.0  


In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),CO (mg/m³),Ozone (µg/m³),Xylene (µg/m³),TOT-RF (mm)
0,2024-01-01,1.433685,1.576067,1.116192,0.533581,0.705624,0.709008,0.276018,-0.126314,0.0
1,2024-01-02,1.511451,1.499319,0.561033,0.271959,0.254182,0.085789,0.257371,-0.126314,0.0
2,2024-01-03,1.633875,1.984529,-0.047478,0.633829,0.099186,0.545003,-0.282208,-0.126314,0.0
3,2024-01-04,1.459015,1.811018,0.471341,0.289890,0.263347,0.380998,-1.051574,-0.126314,0.0
4,2024-01-05,0.419182,0.905036,0.779076,0.280924,0.420734,0.331796,-0.974577,-0.126314,0.0
...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,1.025084,-0.386394,0.648405,-0.111101,-0.412421,-0.635833,0.621902,-1.230459,0.0
362,2024-12-28,-0.178056,-0.621453,-0.829959,-0.464006,-1.124048,-1.800268,0.214662,-1.139261,0.0
363,2024-12-29,-0.306257,-0.967833,-1.051869,-1.031262,-1.365906,-1.997074,-0.176939,-1.217431,0.0
364,2024-12-30,-0.216050,-0.650354,-1.257541,-0.888633,-1.414517,-1.931472,1.367808,-1.165318,0.0


In [10]:
df.to_excel('Ayanagar2024.xlsx', index=False)